# NEXUS-KO · interface **energy** vs co-dependency (AlphaFold-Multimer + PRODIGY, GPU)

The sandbox already **fetched 34 real PDB co-structures** for top obligate pairs (`nexus_ko_fetch.json`) and found that crude interface **size** (residue count) does *not* significantly predict DepMap co-dependency (Spearman 0.19, p=0.29). Binary **contact** carried the obligate signal; size among contacting pairs did not.

This notebook does the honest next step that needs a GPU: for each obligate pair, **predict the complex with AlphaFold-Multimer (ColabFold)**, then compute a **real interface binding energy ΔG with PRODIGY** (not just residue count), plus AF's own interface confidence (ipTM, interface-PAE). Then test whether **interface energy** predicts co-dependency better than size did — and, on the pairs that also have a solved PDB, whether the *predicted* interface matches the *real* one.

**Honest bounds (unchanged):** this is the protein-level *“which complexes break”* readout, validated against co-dependency — **not** the mRNA far field and **not** the perturbation wall. AF-Multimer confidence and PRODIGY ΔG are estimates, not measured affinities.

**Before running:** `Runtime → Change runtime type → GPU`. Predicting all 55 pairs takes a while; `N_PAIRS` below defaults to a manageable subset (no-structure pairs first). 

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached


## 1 · Install ColabFold (AlphaFold-Multimer) + PRODIGY

In [ ]:
# ColabFold (AF-Multimer with the MMseqs2 MSA server) + PRODIGY (structure->binding ΔG). ~3-5 min.
import os, sys
if not os.path.exists('COLABFOLD_READY'):
    os.system('pip -q install "colabfold[alphafold]" 2>/dev/null')
    os.system('pip -q install prodigy-prot biopython scipy 2>/dev/null')
    open('COLABFOLD_READY','w').close()
print('installed. colabfold_batch:', os.popen('which colabfold_batch').read().strip() or 'NOT FOUND -- see ColabFold docs')


## 2 · The obligate pairs (embedded from `nexus_ko_fetch.json`)
`pdb`/`real_iface` are the sandbox-fetched real structure + measured interface (residue count) where one exists.

In [ ]:
PAIRS = [{"a": "TSC1","b": "TSC2","codep": 0.912,"pdb": "7DL2","real_iface": 200},{"a": "DEPDC5","b": "NPRL2","codep": 0.799,"pdb": "6CES","real_iface": 77},{"a": "SDHA","b": "SDHB","codep": 0.794,"pdb": "8GS8","real_iface": 127},{"a": "RNASEH2A","b": "RNASEH2C","codep": 0.789,"pdb": "3P56","real_iface": 120},{"a": "AP2M1","b": "AP2S1","codep": 0.781,"pdb": "6URI","real_iface": 9},{"a": "POLE3","b": "POLE4","codep": 0.776,"pdb": null,"real_iface": null},{"a": "WDR26","b": "YPEL5","codep": 0.762,"pdb": "8QBN","real_iface": 58},{"a": "PDHA1","b": "PDHB","codep": 0.758,"pdb": "1NI4","real_iface": 512},{"a": "HSD17B10","b": "PRORP","codep": 0.757,"pdb": null,"real_iface": null},{"a": "POLG","b": "POLG2","codep": 0.745,"pdb": "3IKM","real_iface": 234},{"a": "MAPKAP1","b": "RICTOR","codep": 0.744,"pdb": "5ZCS","real_iface": 42},{"a": "GATB","b": "QRSL1","codep": 0.736,"pdb": null,"real_iface": null},{"a": "RGP1","b": "RIC1","codep": 0.736,"pdb": null,"real_iface": null},{"a": "MIOS","b": "WDR24","codep": 0.733,"pdb": "7UHY","real_iface": 145},{"a": "KCMF1","b": "UBR4","codep": 0.733,"pdb": "9NWE","real_iface": 202},{"a": "NCAPG2","b": "NCAPH2","codep": 0.729,"pdb": "9F5W","real_iface": 85},{"a": "EED","b": "EZH2","codep": 0.728,"pdb": "5GSA","real_iface": 144},{"a": "BRD9","b": "SMARCD1","codep": 0.727,"pdb": null,"real_iface": null},{"a": "EXT1","b": "EXT2","codep": 0.727,"pdb": "7SCH","real_iface": 181},{"a": "PARD3","b": "PARD6B","codep": 0.726,"pdb": null,"real_iface": null},{"a": "GET1","b": "GET3","codep": 0.726,"pdb": "6SO5","real_iface": 63},{"a": "PSMG1","b": "PSMG2","codep": 0.725,"pdb": "8QYJ","real_iface": 72},{"a": "NCAPD3","b": "NCAPH2","codep": 0.723,"pdb": "9F5W","real_iface": 119},{"a": "DLST","b": "OGDH","codep": 0.706,"pdb": null,"real_iface": null},{"a": "TGFBR1","b": "TGFBR2","codep": 0.704,"pdb": "2PJY","real_iface": 25},{"a": "GATB","b": "GATC","codep": 0.702,"pdb": null,"real_iface": null},{"a": "MAU2","b": "NIPBL","codep": 0.701,"pdb": null,"real_iface": null},{"a": "STN1","b": "TEN1","codep": 0.701,"pdb": "4JOI","real_iface": 191},{"a": "RAD51D","b": "XRCC2","codep": 0.697,"pdb": "8FAZ","real_iface": 108},{"a": "MIOS","b": "WDR59","codep": 0.696,"pdb": "7UHY","real_iface": 130},{"a": "NCAPD3","b": "NCAPG2","codep": 0.694,"pdb": "9F5W","real_iface": 21},{"a": "VPS18","b": "VPS33A","codep": 0.694,"pdb": null,"real_iface": null},{"a": "FANCD2","b": "FANCI","codep": 0.694,"pdb": "6VAA","real_iface": 111},{"a": "METTL1","b": "WDR4","codep": 0.685,"pdb": "7U20","real_iface": 49},{"a": "EED","b": "SUZ12","codep": 0.685,"pdb": "4W2R","real_iface": 62},{"a": "PNPT1","b": "SUPV3L1","codep": 0.685,"pdb": null,"real_iface": null},{"a": "HUS1","b": "RAD9A","codep": 0.683,"pdb": "3A1J","real_iface": 55},{"a": "EZH2","b": "SUZ12","codep": 0.683,"pdb": "5HYN","real_iface": 551},{"a": "ITGAV","b": "ITGB5","codep": 0.678,"pdb": null,"real_iface": null},{"a": "RNASEH2A","b": "RNASEH2B","codep": 0.675,"pdb": "3P56","real_iface": 77},{"a": "MLST8","b": "RICTOR","codep": 0.675,"pdb": null,"real_iface": null},{"a": "SDHB","b": "SDHC","codep": 0.674,"pdb": "8GS8","real_iface": 81},{"a": "PARD6B","b": "PRKCI","codep": 0.669,"pdb": null,"real_iface": null},{"a": "MICOS10","b": "MICOS13","codep": 0.666,"pdb": null,"real_iface": null},{"a": "PRORP","b": "TRMT10C","codep": 0.659,"pdb": "7ONU","real_iface": 45},{"a": "MAEA","b": "WDR26","codep": 0.657,"pdb": null,"real_iface": null},{"a": "CBFB","b": "RUNX1","codep": 0.656,"pdb": "1E50","real_iface": 332},{"a": "COG5","b": "COG7","codep": 0.655,"pdb": null,"real_iface": null},{"a": "ACTR2","b": "ARPC4","codep": 0.654,"pdb": "6UHC","real_iface": 37},{"a": "TUBD1","b": "TUBE1","codep": 0.653,"pdb": null,"real_iface": null},{"a": "CABIN1","b": "HIRA","codep": 0.651,"pdb": null,"real_iface": null},{"a": "SDHA","b": "SDHC","codep": 0.647,"pdb": "8GS8","real_iface": 3},{"a": "VPS39","b": "VPS41","codep": 0.647,"pdb": null,"real_iface": null},{"a": "PEX26","b": "PEX6","codep": 0.645,"pdb": null,"real_iface": null},{"a": "RNASEH2B","b": "RNASEH2C","codep": 0.642,"pdb": "3P56","real_iface": 216}]
N_PAIRS = 20   # predict this many (no-structure pairs first, then with-structure for validation). Raise to 55 for the full set.
pairs_sorted = sorted(PAIRS, key=lambda p: (p['pdb'] is not None, -p['codep']))  # no-structure first, high codep first
work = pairs_sorted[:N_PAIRS]
print(f'predicting {len(work)} pairs; {sum(1 for p in work if not p["pdb"])} have no solved structure')


## 3 · Fetch sequences (UniProt) and write ColabFold multimer FASTAs

In [ ]:
import requests, re, os
os.makedirs('af_in', exist_ok=True)
def gene_seq(g):
    r = requests.get('https://rest.uniprot.org/uniprotkb/search',
                     params={'query':f'gene_exact:{g} AND organism_id:9606 AND reviewed:true','fields':'sequence','format':'fasta'}, timeout=30)
    if r.status_code!=200 or not r.text.startswith('>'): return None
    return ''.join(r.text.split('\n')[1:]).strip()
seqs = {}
for p in work:
    for g in (p['a'], p['b']):
        if g not in seqs: seqs[g] = gene_seq(g)
written = []
for p in work:
    sa, sb = seqs.get(p['a']), seqs.get(p['b'])
    if not sa or not sb or len(sa)+len(sb) > 1800:  # skip missing / too-large for a quick run
        continue
    name = f"{p['a']}__{p['b']}"
    open(f'af_in/{name}.fasta','w').write(f'>{name}\n{sa}:{sb}\n')  # ':' joins chains for AF-Multimer
    written.append((name, p))
print('wrote', len(written), 'multimer FASTAs (skipped missing-seq or >1800 aa)')


## 4 · Run AlphaFold-Multimer (ColabFold)
1 model, 3 recycles for speed. Each pair ~1-5 min on a T4/A100 depending on length.

In [ ]:
import os
os.makedirs('af_out', exist_ok=True)
# one call processes the whole af_in directory; --num-models 1 keeps it fast
os.system('colabfold_batch af_in af_out --num-models 1 --num-recycle 3 --rank iptm 2>&1 | tail -5')
print('done. outputs in af_out/')


## 5 · Interface **energy** (PRODIGY ΔG) + AF confidence (ipTM, interface-PAE)

In [ ]:
import json, glob, numpy as np
from Bio.PDB import PDBParser, NeighborSearch
def top_pdb(name):
    hits = sorted(glob.glob(f'af_out/{name}_*rank_001*.pdb') + glob.glob(f'af_out/{name}*rank_1*.pdb'))
    return hits[0] if hits else None
def scores(name):
    js = sorted(glob.glob(f'af_out/{name}_*rank_001*.json') + glob.glob(f'af_out/{name}*scores*rank_1*.json'))
    if not js: return {}
    d = json.load(open(js[0])); return {'iptm': d.get('iptm'), 'ptm': d.get('ptm'), 'pae': d.get('pae') or d.get('predicted_aligned_error')}
def prodigy_dg(pdb):
    out = os.popen(f'prodigy {pdb} --selection A B -q 2>/dev/null').read()
    m = re.search(r'Predicted binding affinity.*?(-?\d+\.\d+)', out) or re.search(r'(-?\d+\.\d+)\s*$', out.strip())
    try: return float(m.group(1))
    except Exception: return None
import re
rows = []
for name, p in written:
    pdb = top_pdb(name)
    if not pdb: continue
    sc = scores(name); dg = prodigy_dg(pdb)
    # interface PAE: mean PAE across the A/B block (needs chain lengths)
    ipae = None
    try:
        la = len(seqs[p['a']]); pae = np.array(sc['pae'])
        ipae = float((pae[:la, la:].mean() + pae[la:, :la].mean())/2)
    except Exception: pass
    rows.append({**p, 'iptm': sc.get('iptm'), 'interface_pae': ipae, 'prodigy_dG': dg})
    print(f"{p['a']:9s}-{p['b']:9s} codep={p['codep']:.2f}  ipTM={sc.get('iptm')}  ifacePAE={ipae}  ΔG={dg}")
json.dump(rows, open('nexus_ko_afmultimer.json','w'), indent=1)


## 6 · Does interface **energy** predict co-dependency? (the test size failed)

In [ ]:
import numpy as np
from scipy.stats import spearmanr
R = [r for r in rows if r.get('prodigy_dG') is not None]
cd = np.array([r['codep'] for r in R])
for metric, sign in [('prodigy_dG', -1), ('iptm', 1), ('interface_pae', -1)]:
    v = np.array([r[metric] for r in R if r[metric] is not None], float)
    c = np.array([r['codep'] for r in R if r[metric] is not None], float)
    if len(v) >= 6:
        rho, p = spearmanr(sign*v, c)
        print(f'{metric:14s} vs co-dependency: Spearman rho={rho:+.3f} (p={p:.3f}, n={len(v)})   [sign {sign:+d} so higher=stronger interface]')
# validation: predicted interface energy vs the REAL fetched interface size (where a PDB existed)
val = [(r['prodigy_dG'], r['real_iface']) for r in R if r['real_iface']]
if len(val) >= 5:
    a, b = zip(*val); print('\npredicted ΔG vs real fetched interface size:', round(spearmanr(a, b)[0], 3), f'(n={len(val)})')
print('\nInterpretation: if ΔG / ipTM beats the size result (rho~0.19, p=0.29), interface ENERGY sharpens obligate prediction where crude size did not. If not, even the energy does not beat binary contact -> the obligate signal is contact-presence, not interface strength.')


## 7 · Honest bounds
- This validates a **protein-level** knockout consequence (obligate co-failure) against **co-dependency** — **not** the mRNA far field, **not** the perturbation wall.
- AF-Multimer ipTM and PRODIGY ΔG are **estimates**; treat rankings, not absolute kcal/mol, as meaningful.
- Small N and 'top obligate' selection bias apply; raise `N_PAIRS` and add matched non-obligate contacting pairs for a cleaner test.
- Download `nexus_ko_afmultimer.json` and hand it back to the sandbox to fold into the NEXUS-KO record.